# Mixed Precision Training in PyTorch

This notebook explains what mixed precision is, why it is useful, and how to use it correctly in PyTorch.

## What you will learn

1. What mixed precision means.
2. Why lower-precision math can make training faster and use less memory.
3. What `torch.amp.autocast(...)` does.
4. Why `GradScaler` is needed on CUDA.
5. How to write a safe mixed-precision training loop.
6. Why CPU and CUDA use different low-precision dtypes in practice.

## Big picture

Neural network training usually uses `float32`, which is accurate but relatively expensive in memory and compute. Mixed precision keeps the important parts numerically stable while allowing many operations to run in a lower-precision format such as `float16` or `bfloat16`.

This matters because lower precision can:
- reduce memory usage
- increase throughput on supported hardware
- allow larger batch sizes
- speed up training

The word "mixed" is important: not every operation is forced into low precision. PyTorch automatically chooses safe dtypes for many operations inside an autocast region.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## Core idea: what mixed precision actually changes

A training step has three main phases:

1. Forward pass
2. Backward pass
3. Optimizer update

With standard training, these mostly run in `float32`.

With mixed precision, PyTorch tries to run many forward-pass operations in a lower precision, while keeping numerically sensitive behavior stable.

## `autocast`

`torch.amp.autocast(...)` creates a context where PyTorch chooses efficient dtypes for supported operations.

Example idea:
- matrix multiplications and linear layers may run in `float16` on CUDA or `bfloat16` on CPU
- some numerically sensitive operations may stay in `float32`

So `autocast` is not simply "convert everything to low precision". It is more selective than that.

## Why `GradScaler` exists

`float16` has a smaller numeric range than `float32`. During backpropagation, some gradients can become so small that they underflow to zero.

`GradScaler` addresses this by:
1. multiplying the loss by a large scale factor before `backward()`
2. producing larger intermediate gradients during backpropagation
3. unscaling gradients before the optimizer step
4. adjusting the scale dynamically if overflow is detected

Conceptually:

- scaled loss: `scaled_loss = scale * loss`
- backward computes scaled gradients
- before stepping, PyTorch unscales them
- if gradients contain `inf` or `nan`, the optimizer step is skipped and the scale is reduced

This is why `GradScaler` is mainly associated with CUDA `float16` training. In this notebook, the CPU path uses `bfloat16`, which has a wider exponent range and usually does not require gradient scaling.

## Common numerically sensitive operations

When people say an operation is **numerically sensitive**, they usually mean small rounding errors can create noticeably wrong outputs, `inf`, `nan`, or unstable gradients.

Common examples in deep learning:

1. `softmax`
Because it uses exponentials internally. Large inputs can overflow and very small values can underflow.

2. `log_softmax` and cross-entropy style computations
Doing `softmax` first and then `log` manually is usually less stable than using fused PyTorch implementations.

3. `exp`, `log`, and `pow`
These can grow too quickly, shrink too quickly, or become undefined for certain inputs.

4. Division by very small numbers
This can create extremely large outputs.

5. Normalization operations
Examples: batch norm, layer norm, standardization with variance and square root.
These involve division and variance-related math, which can be sensitive.

6. Large reductions
Examples: `sum`, `mean`, variance, and some matrix operations. Rounding error can accumulate across many values.

7. Very small gradients during backward
This is especially important in `float16`, where tiny gradients may underflow to zero.

### Concrete example: naive softmax vs stable softmax

A naive softmax implementation can overflow:

```python
# unstable idea
probs = torch.exp(x) / torch.exp(x).sum(dim=-1, keepdim=True)
```

A safer implementation subtracts the maximum value first:

```python
# stable idea
x_stable = x - x.max(dim=-1, keepdim=True).values
probs = torch.softmax(x_stable, dim=-1)
```

This is part of why AMP does not blindly cast every operation to low precision. PyTorch keeps many sensitive operations in a safer precision path when needed.

In [ ]:
# 1. Pick device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Dummy dataset
num_samples = 200
input_dim = 8
num_classes = 10
batch_size = 10

X = torch.randn(num_samples, input_dim)
y = torch.randint(0, num_classes, (num_samples,))

loader = DataLoader(
    TensorDataset(X, y),
    batch_size=batch_size,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
)

# 3. Simple model
class Net(nn.Module):
    def __init__(self, in_dim=8, hidden=26, out_dim=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)

model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 4. AMP configuration
use_amp = True

if device.type == "cuda":
    amp_dtype = torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
elif device.type == "cpu":
    amp_dtype = torch.bfloat16
    scaler = None
else:
    amp_dtype = None
    scaler = None

print("AMP enabled:", use_amp)
print("autocast dtype:", amp_dtype)
print("using GradScaler:", scaler is not None)

## Small dtype demo

Before training, it helps to see what autocast is doing.

This cell prints the input dtype and the output dtype of a linear layer inside and outside an autocast region.

What to look for:
- outside autocast, outputs are typically `float32`
- inside autocast, supported ops may produce lower-precision outputs
- the exact dtype depends on the device

In [ ]:
demo_layer = nn.Linear(8, 4).to(device)
demo_x = torch.randn(2, 8, device=device)

print("input dtype:", demo_x.dtype)
print("outside autocast output dtype:", demo_layer(demo_x).dtype)

if use_amp and amp_dtype is not None:
    with torch.amp.autocast(device_type=device.type, dtype=amp_dtype):
        demo_out = demo_layer(demo_x)
    print("inside autocast output dtype:", demo_out.dtype)
else:
    print("autocast not enabled for this device")

## Training loop with mixed precision

Read the loop in this order:

1. Move each batch to the target device.
2. Clear old gradients with `optimizer.zero_grad(set_to_none=True)`.
3. Run the forward pass inside `autocast` when AMP is enabled.
4. Compute the loss.
5. If a scaler exists, scale the loss, backpropagate, step the optimizer, and update the scale.
6. Otherwise, use the standard `loss.backward()` and `optimizer.step()` path.

### Why `set_to_none=True`?

This clears gradients by setting them to `None` rather than filling existing buffers with zeros. It is often slightly more memory-efficient and is a common modern default.

### Why not always use `GradScaler`?

On CUDA with `float16`, scaling is important because `float16` can underflow during backward.
On CPU, this notebook uses `bfloat16`, which has a wider exponent range, so gradient scaling is typically unnecessary.

In [ ]:
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=(device.type == "cuda"))
        yb = yb.to(device, non_blocking=(device.type == "cuda"))

        optimizer.zero_grad(set_to_none=True)

        if use_amp and amp_dtype is not None:
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype):
                logits = model(xb)
                loss = criterion(logits, yb)
        else:
            logits = model(xb)
            loss = criterion(logits, yb)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == yb).sum().item()
        total += xb.size(0)

    epoch_loss = running_loss / total
    epoch_acc = running_correct / total
    print(f"Epoch {epoch + 1}/{epochs} | loss={epoch_loss:.4f} | acc={epoch_acc:.4f}")

## Practical interpretation

A few points matter in real training code:

- Mixed precision is helpful only when the hardware supports it well.
- CUDA GPUs often benefit a lot from `float16` AMP.
- CPUs often use `bfloat16` instead of `float16` because `bfloat16` is usually more numerically forgiving.
- `autocast` changes forward-pass precision policy, not the high-level training logic.
- `GradScaler` protects the backward pass from `float16` underflow.

## Mental model

You can think about AMP as:

- `autocast`: "use lower precision where it is safe and profitable"
- `GradScaler`: "protect tiny gradients from disappearing during backward"

Together they make mixed precision practical instead of fragile.

## When would you skip mixed precision?

You may skip it when:
- your hardware gets little benefit from it
- your model is tiny and training is already cheap
- you are debugging numerical issues and want the simplest baseline first